# News2Stock LoRA 파인튜닝

경제 뉴스와 주식 영향 분석 정답으로 `NCSOFT/Llama-VARCO-8B-Instruct`를 지도 미세조정한다.

전체 모델을 다시 학습하는 대신 작은 PEFT adapter만 갱신하므로 제한된 GPU에서도 실습할 수 있다.


## 환경설정


RunPod의 Python 3.12·PyTorch 2.8 환경에 재현 가능한 라이브러리 조합을 설치하고 현재 kernel의 CUDA와 패키지 버전을 확인한다.

PyCharm에서는 SSH interpreter 등록과 별도로 RunPod의 **외부 Jupyter server와 kernel**을 선택해야 Notebook 셀이 원격 GPU에서 실행된다. 연결 과정은 [RunPod SSH·PyCharm 연결](../10_sllm/02_runpod_ssh_pycharm.ipynb)을 참고한다.

Pod 환경변수에 `HF_HOME=/workspace/cache/huggingface`를 지정하면 model과 Dataset cache를 Network Volume에 유지할 수 있다. 공개 데이터와 모델 다운로드에는 `HF_TOKEN`이 필요하지 않으며, token은 선택적 Hub 업로드에만 사용한다. Edit Pod에서 환경변수를 바꿨다면 Pod를 재시작하고 외부 Jupyter server에 다시 연결한다. token 발급과 RunPod Secret 연결은 [Hugging Face 모델·환경변수 준비](../10_sllm/03_sllm.ipynb)를 참고한다.


In [ ]:
%pip install -U "transformers==4.56.2" "accelerate>=1,<2" "trl==0.22.2" "peft==0.17.1" "datasets==3.6.0" python-dotenv hf_transfer


### GPU와 패키지 버전 확인

설치한 라이브러리 버전과 CUDA·GPU 사용 가능 여부를 확인한다. 이 셀의 출력은 이후 8B BF16 모델과 LoRA 학습이 현재 RunPod kernel에서 실행 가능한지 판단하는 기준이 된다.

In [ ]:
%pip install -U torchvision

In [ ]:
import torch
import torchvision
import transformers, datasets, accelerate, trl, peft

print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

import transformers, datasets, accelerate, trl, peft

print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("trl:", trl.__version__)
print("peft:", peft.__version__)


## 데이터 로드와 메시지 변환


News2Stock 원본 1,000건을 입력으로 받아 학습 800건과 평가 200건으로 분할한다. 각 행을 `system → user → assistant` 메시지로 변환하고 Hugging Face Dataset으로 만들어 모델 학습과 평가 단계에 전달한다.

첫 실행에서는 완성된 1,000건 공개 Dataset [`shqkel/naver-economy-news2stock`](https://huggingface.co/datasets/shqkel/naver-economy-news2stock)을 Hugging Face Hub에서 내려받는다. 이 저장소는 03번에서 개인 Hub에 올린 5건 표본과 별개이며, 공개 Dataset이므로 `HF_TOKEN` 없이 다운로드할 수 있다. 내려받은 파일은 `HF_HOME`의 cache에 저장된다.


In [ ]:
import json

from datasets import Dataset, load_dataset

# 허깅페이스에서 데이터셋 다운로드
dataset = load_dataset("shqkel/naver-economy-news2stock")

print(dataset)
print(len(dataset["train"]))


# 학습/평가 데이터 분기
test_ratio = 0.2

train_data, test_data = [], []

data_indices = list(range(len(dataset['train'])))
test_size = int(len(data_indices) * test_ratio)

test_index = data_indices[:test_size] # 0 ~ 199 평가셋
train_index = data_indices[test_size:] # 200 ~ 999 학습셋



import json
# system, user, assistant의 구조로 변환

# assistant 딕셔너리는 작은따옴표의 Python 표현이 아니라 JSON 문자열 계약으로 직렬화한다.
def format_data(data):
    return {
        "messages": [
            {"role": "system", "content": data["system"]},
            {"role": "user", "content": data["user"]},
            {"role": "assistant", "content": json.dumps(data["assistant"], ensure_ascii=False)},
        ]
    }

train_dataset = [format_data(dataset['train'][i]) for i in train_index]
test_dataset = [format_data(dataset['train'][i]) for i in test_index]

print('학습셋:', len(train_dataset))
print('평가셋:', len(test_dataset))

test_dataset[0]

# HuggingFace Dataset으로 변환
from datasets import Dataset

train_dataset = Dataset.from_list(train_dataset)
test_dataset = Dataset.from_list(test_dataset)

## Base model·채팅 형식·LoRA 설정


[`NCSOFT/Llama-VARCO-8B-Instruct`](https://huggingface.co/NCSOFT/Llama-VARCO-8B-Instruct)는 Meta Llama 3.1 8B를 기반으로 한국어·영어 데이터를 추가 학습하고, 한국어 지시문에 맞게 SFT와 DPO를 적용한 모델이다. 여기서 **Base model**은 처음부터 다시 학습할 원시 모델이라는 뜻이 아니라, 금융 뉴스용 LoRA adapter를 붙이기 전의 8B 본체를 뜻한다. 이후 학습에서는 이 본체의 기존 가중치는 유지하고 작은 adapter만 갱신한다.

Dataset의 `messages`는 `system`, `user`, `assistant` 역할을 담은 Python 목록이므로 그대로는 모델 입력 token이 아니다. `tokenizer.apply_chat_template()`는 이 목록을 모델이 학습할 때 사용한 특수 token과 역할 순서가 포함된 하나의 채팅 문자열로 변환한다. 이 실습에서는 `tokenize=False`로 변환된 문자열을 먼저 출력해 구조를 확인하며, 학습과 추론에서 같은 template을 사용해야 입력 형식이 일관된다.

LoRA는 8B 전체 가중치를 다시 학습하는 대신 attention의 `q_proj`와 `v_proj`에 작은 저랭크 행렬을 추가한다. `q_proj`는 현재 token이 어떤 정보에 주목할지 계산하는 query를 만들고, `v_proj`는 주목한 위치에서 가져올 value를 만든다. 이 두 계층의 adapter만 학습하면 전체 fine-tuning보다 학습 파라미터와 VRAM 사용량을 크게 줄이면서 금융 뉴스 분석 형식을 학습시킬 수 있다.

다음 코드는 `messages → 채팅 문자열 → tokenizer 입력 → Base model` 형식을 확인한 뒤, 이후 `SFTTrainer`가 결합할 `LoraConfig`를 만든다. 길이가 다른 문장을 batch로 묶을 때 필요한 padding token이 없을 수 있으므로 EOS token을 padding에도 사용하고, BF16 가중치를 GPU에 적재해 FP32보다 메모리 사용량을 줄인다.

이 셀을 처음 실행하면 공개 8B model 파일을 Hugging Face Hub에서 내려받는다. 다운로드에는 `HF_TOKEN`이 필요하지 않지만 BF16 가중치와 학습용 activation을 함께 GPU에 올리므로 실습에는 A40 48GB급 GPU를 권장한다.


In [ ]:
import torch

from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "NCSOFT/Llama-VARCO-8B-Instruct"

# 사전 학습된 Tokenizer (어휘 사전이 완성 되어 있음)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# 사전 학습된 모델 다운로드
model = AutoModelForCausalLM.from_pretrained(
    model_id,

    # bfloat16: 모델 가중치(실수 좌표값)을 bfloat16 형식으로 로드함.
    # - 기본 float32(32비트)보다 크기가 절반으로 줄어서
    #   GPU 사용량 감소 및 연산 안정성을 유지
    torch_dtype=torch.bfloat16,
    device_map="auto" # GPU 또는 CPU 사용 여부 자동 결정(accelerate)
)

# apply_chat_template()
# - llama 전용 특수 토큰이 포함된 하나의 문자열로 변환

# tokenize=False: token ID가 아니라 사람이 확인 가능한 text 반환
text = tokenizer.apply_chat_template(
    train_dataset[128]["messages"],
    tokenize=False
)

print(text)

In [ ]:
from peft import LoraConfig

#  r은 rank, lora_alpha는 반영 scale, lora_dropout은 과적합 완화, bias는 기존 bias 학습 여부, target_modules는 적용 계층, task_type은 모델 작업 유형을 결정한다.
lora_config = LoraConfig(
    # r은 축소된 내부 차원으로, 작을수록 학습 파라미터가 줄어든다.
    r=8,
    # lora_alpha는 adapter 출력의 반영 크기를 조절하며 현재 scale은 alpha/r인 4이다.
    lora_alpha=32,
    # 학습 중 adapter 입력의 10%를 무작위로 제외해 과적합을 줄인다.
    lora_dropout=0.1,
    # 기존 선형층의 bias는 고정하고 LoRA 행렬만 학습한다.
    bias='none',
    # attention에서 query와 value를 만드는 projection에만 adapter를 연결한다.
    target_modules=['q_proj', 'v_proj'],

    # PEFT가 다음 token 생성용 causal language model 구조를 선택하게 한다.
    task_type='CAUSAL_LM' # text_generation
)

## SFT 학습 설정


모델·데이터·GPU 조건을 `SFTConfig` 하나로 묶는다. `max_length=8192`, BF16, gradient accumulation과 checkpointing을 적용하고 자동 Hub 업로드는 끈 상태로 로컬 저장 경로를 다음 단계에 전달한다.

`output_dir`은 `/workspace/models/news2stock-lora`로 지정한다. PyCharm의 원격 작업 폴더가 `/tmp`여도 adapter와 checkpoint를 Network Volume에 보존하고 다음 추론 단계에서 같은 경로를 사용한다.


In [ ]:
import os

from trl import SFTConfig

# 한 샘플에서 유지할 최대 토큰 수
sequence_max_length = 8192
OUTPUT_DIR = "/workspace/models/news2stock-lora"
HF_TOKEN = os.getenv("HF_TOKEN", "")

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR, # checkpoint와 최종 adapter 저장 로컬 경로
    num_train_epochs=3,
    per_device_train_batch_size=2, # 순전파/역전파에서 처리하는 sample 개수

    # 두 mini-batch의 gradient를 모은 뒤 한 번 갱신하므로 단일 GPU의 실효 batch 크기는 2 × 2 = 4이다.
    gradient_accumulation_steps=2,

    # forward 중간값을 모두 저장하지 않고 backward 때 다시 계산해 VRAM 사용량을 줄이되 계산 시간은 늘어난다.
    gradient_checkpointing=True,

    # PyTorch의 CUDA fused AdamW를 사용해 optimizer 연산을 묶어 처리
    optim="adamw_torch_fused",

    # 10번의 optimizer 갱신마다 loss와 학습률 등의 학습 로그 출력
    logging_steps=10,

    # epoch가 아니라 optimizer step 수를 기준으로 checkpoint를 저장함
    save_strategy="steps",

    # save_strategy가 steps이므로 50번의 optimizer 갱신마다 checkpoint를 저장함.
    save_steps=50,

    # FP32보다 메모리를 적게 쓰는 BF16 정밀도로 학습
    bf16=True,

    # LoRA adapter 가중치를 한 번 갱신할 때 적용할 기본 학습률을 0.0001 지정
    learning_rate=1e-4,

    # 전체 gradient의 norm이 0.3을 넘으면 잘라 급격한 가중치 변화를 억제함
    max_grad_norm=0.3,

    # 전체 학습 step의 앞 3% 동안 학습률을 목표값까지 서서히 높인다.
    warmup_ratio=0.03,

    # warmup이 끝난 뒤에는 학습률을 1e-4로 일정하게 유지
    lr_scheduler_type="constant",

    # 학습 결과를 Hugging Face Hub에 자동 업로드하지 않고 output_dir에만 저장
    push_to_hub=False,

    hub_token=HF_TOKEN,

    # Trainer가 model.forward()에 없는 messages 열을 제거하지 않게 해 custom collator까지 전달
    remove_unused_columns=False,

    # TRL의 자동 tokenization을 건너뛰고 아래 data_collator_fn이 messages를 직접 처리하게 한다.
    dataset_kwargs={"skip_prepare_dataset": True},

    # Weights & Biases나 TensorBoard 같은 외부 실험 추적 서비스로 기록을 보내지 않는다.
    report_to=[],

    # SFTConfig와 custom collator가 공유하는 최대 입력 길이를 8192 token으로 맞춤
    max_length=sequence_max_length,

    # custom collator가 만든 labels key를 loss 계산에 사용할 정답 tensor 이름으로 지정함
    label_names=["labels"],


)

## Data Collator와 batch 확인


Data Collator는 Dataset에서 꺼낸 여러 sample을 모델이 한 번에 받을 수 있는 batch tensor로 조립하는 함수이다. 이 실습에서는 `messages` 대화를 Llama 3 채팅 문자열로 연결하고 token ID로 바꾼 뒤, 길이가 다른 sample을 현재 batch에서 가장 긴 길이에 맞춰 padding한다. 반환된 딕셔너리는 이후 `SFTTrainer`가 모델에 전달한다.

- `input_ids`는 대화의 `system`·`user`·`assistant` 내용과 Llama 특수 token을 정수 ID로 변환한 모델 입력이다. 실제 대화의 흐름을 모두 모델에 보여 주며, 짧은 sample의 뒤에는 `tokenizer.pad_token_id`가 추가된다.
- `attention_mask`는 모델이 읽을 위치를 구분하는 값이다. 실제 token은 `1`, 길이를 맞추기 위해 추가한 padding은 `0`이므로 Transformer가 padding 위치를 문맥으로 사용하지 않는다.
- `labels`는 다음 token을 맞히도록 모델에 제공하는 학습 정답이다. assistant 답변과 답변 종료 token에는 해당 `input_ids` 값을 넣고, `system`·`user`·padding 위치에는 `-100`을 넣는다. PyTorch의 loss 계산은 `-100` 위치를 무시하므로 질문을 외우게 하지 않고 답변을 생성하는 구간만 학습할 수 있다.

예제에서는 `train_dataset`의 두 sample을 전달하므로 세 tensor의 첫 번째 축은 batch 크기 `2`이다. 두 번째 축은 두 sample 중 더 긴 token 길이에 맞춰지며 `sequence_max_length=8192`를 넘지 않는다. 실행 후 `input_ids`, `attention_mask`, `labels`가 모두 `(batch 크기, token 길이)`라는 같은 shape인지 확인한다.


In [ ]:
# assistant 답변만 학습하는 collator를 만들고 batch shape를 확인한다.

# 목표: 학습용 입력 샘플을 읽고,
# assistant가 작성한 답변 부분만 채점에 활용할 수 있도록
# 미니배치(batch)를 만드는 함수
# == 입력 값에 대한 채점용 batch 생성 함수
def data_collator_fn(batch):

    # 미니배치(batch) 데이터를 저장할 딕셔너리
    # "input_ids": 각 샘플의 숫자 토큰 시퀀스(모델 입력)
    # "attention_mask":	input_ids에서 실제 데이터는 1, 패딩은 0으로 구분(모델이 무시해야 할 부분 표시)
    # "labels":	학습 정답(정답이 아닌 위치는 -100, assistant(답변) 구간만 정답 토큰 값으로 채움)
    new_batch = {
        "input_ids": [],
        "attention_mask": [],
        "labels": []
    }

    # 미니배치(batch) 안에 있는 각 예시 데이터를 하나씩 처리
    for example in batch:
        # 예시 데이터에서 메시지 리스트(system, user, assistant 등)를 꺼냄.
        messages = example["messages"]

        # LLaMA 3 채팅 템플릿 적용 (시작 토큰 포함)
        # 전체 프롬프트 텍스트의 시작에 특별 토큰을 넣음. (LLaMA-3 채팅 포맷에서 전체 대화의 시작을 알림)
        prompt = "<|begin_of_text|>"

        for msg in messages:   # 각 메시지(시스템, 유저, 어시스턴트 등)에 대해 아래 작업을 반복

            role = msg["role"]      # 메시지의 역할(예: "system", "user", "assistant")을 꺼냄.

            content = msg["content"].strip()    # 메시지 본문(내용)을 꺼내고, 앞뒤 공백을 제거.

            prompt += f"<|start_header_id|>{role}<|end_header_id|>\n{content}<|eot_id|>"
            # 각 메시지를 LLaMA-3 채팅 포맷에 맞춰 특별 토큰으로 감싸 하나의 프롬프트 문자열로 이어붙임.
            # <|start_header_id|>역할<|end_header_id|> 각 메시지의 역할(시스템/유저/어시스턴트) 표시
            # \n내용<|eot_id|> 실제 메시지 내용과 그 끝을 나타내는 토큰

        prompt = prompt.strip()   # 완성된 전체 프롬프트 텍스트의 앞뒤 공백을 제거해서 최종적으로 저장


        # 프롬프트 전체 텍스트를 모델이 이해할 수 있는 숫자 토큰(input_ids)으로 변환.
        tokenized = tokenizer(
            prompt,
            truncation=True,             # 입력이 sequence_max_length(최대 길이)를 넘으면 자동으로 잘라냄.
            max_length=sequence_max_length,   # 입력 토큰의 최대 개수를 제한.
            padding=False,               # 이 단계에서는 패딩하지 않고 나중에 미니배치의 최대 길이에 맞춘다.
            return_tensors=None,         # 결과를 일반 파이썬 리스트 형태로 반환.
        )

        # 텍스트가 숫자 토큰 리스트로 바뀐 결과. 예: [128, 5551, 29871, ...] (각 숫자는 단어나 특수토큰에 해당)
        input_ids = tokenized["input_ids"]

        # input_ids에서 실제 데이터(=1), 패딩(=0)을 구분하는 마스크.여기선 모두 1로만 채워짐(아직 패딩이 없으므로).
        attention_mask = tokenized["attention_mask"]

        # input_ids와 동일한 길이의 리스트를 -100으로 채움. -100은 PyTorch에서 "이 위치는 손실 계산(학습)에서 무시하라"는 의미임.
        labels = [-100] * len(input_ids)
        # 이후 assistant(답변) 구간에서만 실제 정답 토큰값으로 바뀜.


        # 답변(assistant) 부분이 시작되는 지점의 특수 토큰 문자열. 이 토큰 뒤부터 모델의 정답(레이블)로 사용할 구간이 시작
        # ex) assitant 대답
        # assistant_header 문자열을 숫자 토큰 시퀀스로 변환. 나중에 input_ids 안에서 이 시퀀스가 어디에 있는지 찾아 "정답 시작 위치"로 사용.
        assistant_header = "<|start_header_id|>assistant<|end_header_id|>\n"
        assistant_tokens = tokenizer.encode(assistant_header, add_special_tokens=False)

        # print("assistant_tokens: ", assistant_tokens)
        # 한 메시지(assistant 답변)가 끝났음을 표시하는 특수 토큰 문자열
        # eot_token도 숫자 토큰 시퀀스로 변환. 나중에 답변의 끝 위치를 정확히 찾는 데 사용.
        eot_token = "<|eot_id|>"
        eot_tokens = tokenizer.encode(eot_token, add_special_tokens=False)
        # print("eot_token: ", eot_tokens)


        # input_ids(토큰 시퀀스)에서 assistant(정답) 구간만 정확하게 찾아서 labels에 복사해 모델이 질문-답변 데이터에서“정답(답변 부분)만 학습”하도록 레이블을 세팅.
        i = 0
        # input_ids 리스트에서 assistant_tokens(=assistant 시작 토큰 시퀀스)가 어디에 있는지 찾기
        while i <= len(input_ids) - len(assistant_tokens):

            # 현재 위치(i)부터 assistant 시작 토큰 시퀀스와 정확히 일치하는 부분을 찾으면 아래 실행
            if input_ids[i:i + len(assistant_tokens)] == assistant_tokens:


                # assistant 답변의 "실제 내용"이 시작되는 토큰 위치를 저장.
                start = i + len(assistant_tokens)

                # start부터 eot_tokens(=답변 끝 토큰 시퀀스)가 처음 나올 때까지 end를 증가시켜 답변의 끝 위치를 찾음.
                end = start

                while end <= len(input_ids) - len(eot_tokens):
                    if input_ids[end:end + len(eot_tokens)] == eot_tokens:
                        break
                    end += 1

                # 답변(assistant)의 본문 구간을 labels에 복사해서 정답으로 사용. 이 구간만 손실 계산(모델 학습)에 실제로 반영.
                for j in range(start, end):
                    labels[j] = input_ids[j]

                # eot_tokens(=답변 종료 특수토큰)도 정답에 포함. 모델이 어디서 답변을 끝내야 하는지도 학습
                for j in range(end, end + len(eot_tokens)):
                    labels[j] = input_ids[j]

                # 첫 번째 assistant 구간만 처리하고, 그 뒤는 무시(중복 적용 방지).
                break

            # 다음 위치로 이동하며 assistant 시작 시퀀스를 계속 탐색.
            i += 1
        # print(labels)

        # 이 샘플의 input_ids, attention_mask, labels를 배치(new_batch)에 저장.
        new_batch["input_ids"].append(input_ids)
        new_batch["attention_mask"].append(attention_mask)
        new_batch["labels"].append(labels)

    # 패딩 처리
    max_length = max(len(ids) for ids in new_batch["input_ids"])              # 미니배치 안에서 가장 긴 input_ids의 길이를 구함. 배치 내 모든 입력이 이 길이에 맞게 통일될 예정.
    for i in range(len(new_batch["input_ids"])):                              # 배치의 각 샘플에 대해 아래 과정을 반복
        pad_len = max_length - len(new_batch["input_ids"][i])                 # 현재 샘플의 길이가 max_length보다 짧으면, 부족한 만큼 pad_len을 계산..

        # input_ids의 끝에 pad_token_id(패딩 토큰)를 pad_len만큼 추가해서 길이를 맞춤.
        new_batch["input_ids"][i].extend([tokenizer.pad_token_id] * pad_len)

        # attention_mask의 끝에도 0을 pad_len만큼 추가.(패딩된 부분은 0, 실제 데이터는 1).
        new_batch["attention_mask"][i].extend([0] * pad_len)

        # labels의 끝에는 -100을 pad_len만큼 추가.(패딩된 부분은 학습에서 무시).
        new_batch["labels"][i].extend([-100] * pad_len)

    # 텐서 변환
    for k in new_batch:
        # 각 리스트를 PyTorch 텐서로 변환해서 모델에 바로 입력할 수 있게 만듦.
        new_batch[k] = torch.tensor(new_batch[k])

    return new_batch


# 배치별 고정길이 확인
batch = data_collator_fn([train_dataset[2], train_dataset[3]])

print(batch['input_ids'].shape)
print(batch['attention_mask'].shape)
print(batch['labels'].shape)

## LoRA 학습과 로컬 저장


이 단계에서는 앞에서 준비한 객체를 `SFTTrainer`에 연결해 LoRA adapter를 학습하고 로컬에 저장한다.

### 학습에 연결하는 객체

- `model`: LoRA adapter를 결합할 base model이다.
- `sft_config`: epoch, batch 크기, 학습률과 저장 경로를 담은 학습 설정이다.
- `train_dataset`: `messages` 형식으로 구성된 800개의 학습 sample이다.
- `data_collator_fn`: 여러 sample을 `input_ids`, `attention_mask`, `labels` batch로 변환한다.
- `lora_config`: adapter를 붙일 계층과 rank 등 LoRA 구조를 지정한다.

### 실행과 저장 순서

1. `SFTTrainer(...)`가 모델·설정·Dataset·collator·LoRA 설정을 하나의 학습기로 묶는다.
2. `trainer.train()`이 800개 sample을 3 epoch 동안 반복하며 adapter 가중치를 학습한다.
3. `trainer.save_model(...)`이 학습한 adapter를 `/workspace/models/news2stock-lora`에 저장한다.
4. `tokenizer.save_pretrained(...)`가 추론에 필요한 tokenizer 설정을 같은 경로에 저장한다.

### Hugging Face token 사용 조건

- 공개 base model을 내려받는 데는 token이 필요하지 않다.
- `HF_TOKEN`은 자신의 Hub 저장소로 올리는 주석 처리된 `push_to_hub()` 두 줄을 실행할 때만 필요하다.


In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model         = model, # NCSOFT/llama-3.1-8B
    args          = sft_config, # STF 설정 객체
    train_dataset = train_dataset, # 학습용 dataset
    data_collator = data_collator_fn, # batch, labels 변환
    peft_config   = lora_config # PEFT 중 LoRA 적용 설정
)

train_result = trainer.train() # 학습 수행
trainer.save_model(sft_config.output_dir) # 모델 저장
tokenizer.save_pretrained(sft_config.output_dir) # 토크나이저 저장


## 추론 모델 로드


학습용 Trainer와 base model 참조를 해제해 GPU 메모리를 확보한다. 로컬 adapter 디렉터리를 `AutoPeftModelForCausalLM`으로 다시 읽고 tokenizer와 함께 text-generation pipeline으로 결합한다.


In [ ]:
import gc
import torch
from transformers import AutoTokenizer, pipeline
from peft import AutoPeftModelForCausalLM

# 학습 모델 참조를 해제해 추론 모델을 적재할 VRAM을 확보한다.
del trainer, model
gc.collect()
torch.cuda.empty_cache()

peft_model_id = sft_config.output_dir
inference_tokenizer = AutoTokenizer.from_pretrained(peft_model_id)
inference_model = AutoPeftModelForCausalLM.from_pretrained(
    peft_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

text_generator = pipeline(
    "text-generation",
    model=inference_model,
    tokenizer=inference_tokenizer,
)

## 평가 prompt 준비와 5건 비교


평가 messages에서 assistant 직전까지를 생성 prompt로, assistant 본문을 정답 label로 분리한다. Greedy 생성 함수로 5건을 추론하여 예측과 정답을 나란히 출력하고 형식과 내용 품질을 확인한다.


In [ ]:
prompt_lst = []
label_lst = []

# messages 한 건은 system 지시문, user 뉴스, assistant 정답을 순서대로 담은 목록이다.
for messages in test_dataset["messages"]:
    # apply_chat_template은 역할 목록을 Llama가 학습한 특수 token 형식의 문자열로 합친다.
    # tokenize=False이므로 token ID가 아니라 내용을 눈으로 확인하고 나눌 수 있는 문자열을 반환한다.
    text = inference_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

    # text에는 정답까지 들어 있으므로 그대로 모델에 주면 정답이 누출된다.
    # assistant header 앞부분만 남기고 빈 assistant header를 다시 붙여 모델이 그 뒤부터 답하도록 만든다.
    # 결과 구조: input = system + user + assistant header, label = 원래 assistant 답변이다.
    input = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[0] + \
        '<|start_header_id|>assistant<|end_header_id|>\n'

    # assistant header 다음부터 대화 차례 종료 token인 <|eot_id|> 전까지가 비교할 정답이다.
    label = text.split('<|start_header_id|>assistant<|end_header_id|>\n')[1].split('<|eot_id|>')[0]

    # 두 목록에 같은 순서로 추가해 이후 zip()이 올바른 입력과 정답을 묶게 한다.
    prompt_lst.append(input)
    label_lst.append(label)

# 101번째 샘플로 모델이 실제로 받을 prompt와 숨겨 둔 label의 경계를 확인한다.
print(prompt_lst[100])

print(label_lst[100])

# <|eot_id|>는 assistant 답변 한 차례가 끝났음을 나타내는 특수 token이다.
# encode() 결과는 token ID 목록이며 이 특수 token은 하나이므로 [0]으로 ID 하나를 꺼낸다.
eos_token_id = inference_tokenizer.encode('<|eot_id|>', add_special_tokens=False)[0]

def test_inference(text_generator, prompt):
    # text_generator는 앞에서 만든 pipeline이고 prompt 뒤에 모델의 새 답변을 이어 생성한다.
    # max_new_tokens는 새 답변의 최대 길이, eos_token_id는 정상 종료 지점을 지정한다.
    # do_sample=False는 매 단계 가장 확률이 높은 token을 고르는 greedy 생성으로 결과를 일정하게 만든다.
    outputs = text_generator(
        prompt,
        max_new_tokens=1024,
        eos_token_id=eos_token_id,
        do_sample=False,
    )

    # pipeline의 generated_text는 기본적으로 원래 prompt와 새 답변을 합친 전체 문자열이다.
    # prompt의 문자 수만큼 앞부분을 잘라 모델이 새로 생성한 assistant 답변만 반환한다.
    assistant_start = len(prompt)
    return outputs[0]['generated_text'][assistant_start:].strip()

# Python slice의 끝 index는 포함되지 않으므로 [10:15]는 index 10~14의 다섯 건이다.
start = 10
end = 15

# zip()으로 같은 샘플의 prompt와 label을 묶고 pred(모델 예측)와 label(정답)을 나란히 비교한다.
for prompt, label in zip(prompt_lst[start:end], label_lst[start:end]):
    print(f'pred: \n{test_inference(text_generator, prompt)}')
    print(f'label: \n{label}')
    print('-' * 100)

## 새 뉴스 원문 추론


새 기사 원문을 학습 데이터와 같은 금융 분석 system 지침으로 감싸 생성 prompt로 변환한다. 조선업 기사와 뉴욕 증시 기사에 대해 새 assistant 응답만 잘라 반환하여 일반화 결과를 확인한다.


In [ ]:
def inference(news):
    messages = [
        {'role': 'system', 'content': '''당신은 금융/경제 뉴스의 핵심내용을 요약해 설명하고,
특정 상장된 종목(주식)에 미치는 긍정/부정 영향여부, 이유/근거 등을 분석하는 AI 금융분석 전문가입니다.'''},
        {'role': 'user', 'content': news}
    ]
    prompt = inference_tokenizer.apply_chat_template(messages, tokenize=False)

    # 인자: max_new_tokens는 생성 상한, eos_token_id는 답변 종료 토큰, do_sample은 greedy 생성 여부를 정한다.
    outputs = text_generator(
        prompt,
        max_new_tokens=1024, # 출력토큰에 대한 제한 (max_length: 입출력 전체)
        eos_token_id=eos_token_id,
        do_sample=False, # False: Greedy방식 작동(확률이 가장 토큰 선택. 일관된 답변)
    )
    assistant_start = len(prompt)
    return outputs[0]['generated_text'][assistant_start:].strip()

# 입력: 학습에 포함되지 않은 여러 문단의 경제 기사 원문을 사용한다.
# 변환: inference 함수가 기사 전체를 채팅 prompt로 바꾸고 새 assistant 분석을 생성한다.
inference(news="""
세계적으로 선박 수주량이 주춤한 가운데 광둥(廣東)성 조선업은 오히려 활기를 보이고 있다.

​

광둥성 선박공업협회 통계에 따르면 올해 1~5월 조선 완공량은 전년 동기 대비 2.5% 증가했으며 수주잔량은 29.3% 늘었다.

​

지난해 신규 수주량은 634만7천DWT(재화중량톤수·선박에 적재할 수 있는 최대량)로 전년 동기 대비 60.7% 증가했다. 기존 수주량도 1천127만6천DWT에 달해 37.1% 확대됐다.

​

올 들어 중국선박그룹 산하 광촨(廣船)국제와 황푸원충(黃埔文衝)선박 두 회사의 수주량이 모두 폭발적으로 증가했다.

​

광촨국제는 지난달 10일 한국해양진흥공사(KOBC)가 발주한 액화천연가스(LNG) 이중연료 자동차운반선(PCTC) 건조에 착수했다. 해당 선박은 현존하는 세계 최다 자동차를 운반할 수 있는 이중연료 PCTC 중 하나다. 전체 길이가 230m에 달한다.


광촨(廣船)국제회사가 덴마크 해운회사 DFDS에 납품한 호화 크루즈선. (사진/신화통신)

광촨국제 관계자는 기존 수주량 90여 척 가운데 80%가 하이테크, 고부가가치 기반의 신형 친환경 선박이며 수주 물량이 2028년까지 차 있다고 설명했다.

​

황푸원충선박도 지난달 13일 2만5천㎥의 LPG/액체 암모니아 운반선 착공식을 가졌다. 선박이 완공되면 중국 최초 이중연료 동력의 LPG/액체 암모니아 운반선으로 이름을 올리게 된다.

​

황푸원충선박 측 관계자는 "회사가 수주한 가스운반선이 이미 16척에 달한다"며 "중소형 가스운반선 분야에서 자체 기술을 확보했고 향후 신에너지 선박 시장에서 계속 역량을 키워나갈 계획"이라고 소개했다.

​

광저우(廣州)해양엔지니어링선박설비는 10여 년간의 연구 끝에 세계적으로 선진 수준을 자랑하는 선박 전력 추진 시스템인 무축 림구동 추진기를 개발했다. 무축 림구동 추진기가 기존의 프로펠러 추진기를 대체한다는 것은 비행기의 제트 엔진이 나선형 엔진을 대체한 것과 같은 의미다.

​

추샹야오(邱湘瑤) 광저우해양엔지니어링선박설비 회장은 "올 들어 주문이 폭주해 생산 일정이 1년 후까지 차있다"고 밝혔다. 이어 ㎿(메가와트)급 추진기 수출로 국제 독점 구도를 타파했을 뿐만 아니라 해외 시장 확대 및 원양어업 가공선 후속 사업을 위한 안정적인 주문을 확보했다고 설명했다.


광저우해양엔지니어링선박설비 엔지니어들이 ㎿(메가와트)급 추진기를 개발하고 있다. (사진/신화통신)

천젠룽(陳建榕) 광둥성 선박공업협회 비서장은 중국 3대 조선 기지 중 하나인 광저우는 선박 제조업 업∙미들∙다운스트림을 모두 아우르는 산업망을 갖춰 중국 현대 선박공업을 든든히 뒷받침한다고 소개했다.
[출처] 세계 조선업 수주 주춤...中 광둥성 '나홀로 활황'|작성자 차이나랩""")

# 입력: 학습에 포함되지 않은 여러 문단의 경제 기사 원문을 사용한다.
# 변환: inference 함수가 기사 전체를 채팅 prompt로 바꾸고 새 assistant 분석을 생성한다.
inference(news="""
(뉴욕=연합뉴스) 진정호 연합인포맥스 특파원 = 뉴욕증시의 3대 주가지수가 지루한 흐름을 보인 끝에 보합권에서 혼조로 마감했다.

도널드 트럼프 미국 대통령이 관세 부과 시점을 8월 1일 이후로는 연장하지 않겠다고 공언했으나 그가 숱하게 말을 번복해왔던 만큼 시장은 크게 개의치 않았다.

트럼프는 또 구리에 50%의 관세를 부과하겠다고 밝혔으나 이 또한 예상된 재료였던 만큼 투심을 흔들지는 못했다.

뉴욕증권거래소
[연합뉴스 자료사진]
뉴욕증권거래소
[연합뉴스 자료사진]


8일(미국 동부시간) 뉴욕증권거래소(NYSE)에서 다우존스30산업평균지수는 전장보다 165.60포인트(0.37%) 내린 44,240.76에 거래를 마감했다.

스탠더드앤드푸어스(S&P)500지수는 전장보다 4.46포인트(0.07%) 떨어진 6,225.52, 나스닥종합지수는 5.95포인트(0.03%) 오른 20,418.46에 장을 마쳤다.

트럼프는 이날도 관세 관련 발언을 쏟아냈으나 증시도 내성이 생긴 듯 보합권에서 한산한 움직임을 보였다.

트럼프는 자신의 소셜미디어 트루스소셜에 게시한 글에서 "관세는 2025년 8월 1일부터 부과되기 시작할 것"이라며 "(기한) 연장은 허용되지 않을 것"이라고 말했다.

이는 전날 트럼프가 내놓은 발언과 배치되는 것이다. 트럼프는 전날 한국과 일본 등 14개국에 관세 서한을 보내는 한편 관세 부과 시점을 8월 1일로 연기했으나 협상 상대방이 좋은 제안을 가져오면 관세 부과 시점이 더 미뤄질 수 있다고 말한 바 있다.

트럼프는 또 이르면 이달 말 반도체와 의약품 등 주요 품목에 대해 관세를 부과할 계획이라는 점을 알렸다. 반도체에 대해선 구체적인 관세율과 부과 시점 등이 발표되지 않았으나 의약품은 최대 200%의 관세가 부과될 수 있다고 그는 말했다.

뱅크오브아메리카의 안토니오 가브리엘 이코노미스트는 "전날 발표된 관세가 확정된 것은 아니라고 본다"면서도 "관세가 시행된다면 물가상승률은 약 0.1%포인트 상승하고 성장률은 비슷한 수준으로 하락할 것"이라고 말했다.

트럼프가 구리에 50%의 관세를 부과하기로 한 점은 장기적으로 인플레이션을 자극할 수 있다는 관측도 나온다.

구리는 제조업 전반에 소요되는 필수 요소인 만큼 관세발 인플레이션에도 취약할 수밖에 없다. 트럼프의 발표 이후 금속선물거래소 코멕스(COMEX)에서 구리선물 가격은 한때 17% 폭등하며 역대 최고치를 경신했다.

리베르타스웰스매니지먼트의 아담 쿠스 대표는 "우리는 미국 산업을 보호하기 위해 정책을 무기화하는 움직임을 보고 있지만 이는 인플레이션 공포를 부채질할 것"이라며 "관세 위협이 공식 정책이 되면 힘을 발휘할 수 있겠지만 대부분의 정치 랠리가 그렇듯 짧은 도화선이 될 가능성이 크다"고 말했다.

업종별로는 에너지가 2.72% 급등했고 유틸리티와 필수소비재는 1% 이상 하락했다.

시가총액 1조달러 이상의 거대 기술기업 중에선 엔비디아와 테슬라가 1% 이상 상승했다.

엔비디아는 이날 강세로 시총이 3조9천억달러를 넘어서며 사상 최초 4조달러를 눈앞에 두게 됐다.

엔비디아에 대한 기대감이 반도체 업계 전반으로 퍼지면서 필라델피아 반도체지수도 1.80% 뛰었다. 해당 지수를 구성하는 30개 종목 중 27개가 강세였다.

트럼프가 친환경 에너지 보조금 축소를 골자로 한 행정명령에 서명했다는 소식에 에너지 관련주가 급등했다.

셰브런은 3.96%, 엑손 모빌은 2.77% 상승했다.

반면 태양광 관련주들은 일제히 약세였다. 선런의 주가는 전일 대비 11%, 퍼스트 솔라는 6% 넘게 떨어졌다.

은행주들 역시 이날 약세였다. 은행권의 2분기 실적 시즌을 앞두고 HSBC가 대형 은행에 대한 투자의견을 하향 조정한 여파다.

JP모건체이스와 뱅크오브아메리카의 주가는 3% 넘게 떨어졌고 모건스탠리와 골드만삭스도 2% 가까이 하락했다.

시카고상품거래소(CME) 페드워치툴에 따르면 연방기금금리 선물시장은 7월 기준금리 동결 확률을 95.3%로 유지했다. 연말까지 2회 금리 인하될 확률은 43.7%로 반영되며 가장 가능성이 높게 점쳐졌다.

시카고옵션거래소(CBOE) 변동성 지수(VIX)는 0.98포인트(5.51%) 떨어진 16.81을 기록했다.
""")

### 완성된 모델 허깅페이스에 올리기

학습한 LoRA adapter와 tokenizer를 자신의 Hugging Face 모델 저장소에 업로드한다. 저장소 ID와 쓰기 권한 token은 환경변수로 준비하고, 업로드 코드는 직접 작성한다.


In [ ]:
HF_MODEL_REPO_ID = "goat-baek/news2stock-lora"

# LoRA adapter 업로드
inference_model.push_to_hub(
    HF_MODEL_REPO_ID,
    token=HF_TOKEN,
)

# tokenizer 업로드
inference_tokenizer.push_to_hub(
    HF_MODEL_REPO_ID,
    token=HF_TOKEN,
)

print("업로드 완료:", HF_MODEL_REPO_ID)